# 🎮 Pokémon TCG — Agent Performance Dashboard

One place to review how your submitted agent is doing:

1. **Submissions** — list your submissions and pick one.
2. **Agent KPIs** — win rate, prizes, damage, turns, and win-rate by opponent archetype.
3. **Episodes table** — every match: won/lost, opponent archetype, opponent ELO/score, and per-match KPIs (turns, prizes, KOs, damage).
4. **Watch a replay** — type an episode number and scrub the full battle inline.

All the heavy lifting lives in three local modules so the notebook stays short:

| module | what it does |
|---|---|
| [`kaggle_cli.py`](kaggle_cli.py) | wrappers around the `kaggle` CLI (submissions, episodes, replay/log download) |
| [`replay_analysis.py`](replay_analysis.py) | parse replay JSON → per-match KPIs, archetype classification, aggregate stats |
| [`replay_render.py`](replay_render.py) | self-contained replay viewer (extracted from the visualizer notebook) |

**Prereqs:** `pip install kaggle pandas`, a token at `~/.kaggle/kaggle.json`, and having accepted the competition rules. Card names/sprites in replays come from `card_data/EN_Card_Data.csv` (already extracted from the competition zip in this folder).


In [1]:
import importlib
import pandas as pd

import kaggle_cli as kc
import replay_analysis as ra
import replay_render as rr
for _m in (kc, ra, rr):        # pick up edits to the util modules without restarting
    importlib.reload(_m)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# --- config ---------------------------------------------------------------
REPLAY_DIR = "./replays/json"  # where episode replay JSONs are cached
HTML_DIR   = "./replays/html"  # where rendered replay viewers (.html) are written
MY_TEAM    = "AlphaKarpZero"   # your team name (match against replay TeamNames);
                               # leave None to default to seat 0 (usually fine)
MAX_EPISODES_TO_ANALYZE = 30   # cap replay downloads for the KPI table
print("Ready. Competition:", kc.COMPETITION)

Ready. Competition: pokemon-tcg-ai-battle


## 1. Submissions

Lists all your submissions, most recent first. Pick one to analyze — by default we take the newest.

In [ ]:
submissions_df = kc.list_submissions(echo=False)
display(submissions_df)

# Choose which submission to analyze (override SUBMISSION_ID manually if you like).
SUBMISSION_ID = 54437175
if isinstance(submissions_df, pd.DataFrame) and not submissions_df.empty:
    _idc = kc.pick_id_column(submissions_df)
    SUBMISSION_ID = submissions_df.iloc[0][_idc]
    # If MY_TEAM wasn't set, try to read it from the submissions table.
    for _c in ("teamName", "team", "submittedBy"):
        if MY_TEAM is None and _c in submissions_df.columns:
            MY_TEAM = str(submissions_df.iloc[0][_c])
print("Analyzing SUBMISSION_ID =", SUBMISSION_ID, "| MY_TEAM =", MY_TEAM)

,ref,fileName,date,description,status,publicScore,privateScore
0,54437175,v1.tar.gz,2026-07-07 18:15:32.440000,NaN,SubmissionStatus.COMPLETE,803.1,NaN


Analyzing SUBMISSION_ID = 54437175 | MY_TEAM = AlphaKarpZero


## 2. Agent KPIs

Lists the submission's episodes, downloads their replays (capped by `MAX_EPISODES_TO_ANALYZE`), and extracts per-match KPIs. The first run downloads replays and can take a while; re-runs reuse the cached JSONs in `./replays`.

In [ ]:
episodes_list_df = kc.list_episodes(SUBMISSION_ID, echo=False) if SUBMISSION_ID is not None else pd.DataFrame()
print(f"Episodes reported for this submission: {len(episodes_list_df)}")
display(episodes_list_df.head())

# Clean integer episode ids (drops the CLI's trailing help/footer line).
episode_ids = kc.episode_ids_from_df(episodes_list_df, limit=MAX_EPISODES_TO_ANALYZE)
print(f"Downloading / loading {len(episode_ids)} replays into {REPLAY_DIR} ...")

for _eid in episode_ids:
    try:
        kc.get_replay_path(_eid, dest=REPLAY_DIR, echo=False)
    except Exception as _e:
        print(f"  [skip] episode {_eid}: {_e}")

In [ ]:
# Parse every cached replay into a per-match KPI table.
kpi_df = ra.analyze_folder(REPLAY_DIR, my_team=MY_TEAM)
kpi_df = ra.merge_episodes_metadata(kpi_df, episodes_list_df)   # add ELO/score if present
print(f"Parsed {len(kpi_df)} replays.")

stats = ra.agent_stats(kpi_df)
display(pd.DataFrame([stats]))

In [ ]:
# Quick visuals: result split + win rate by opponent archetype.
import matplotlib.pyplot as plt

by_opp = ra.winrate_by_opponent(kpi_df)
if not by_opp.empty:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    counts = kpi_df["result"].value_counts()
    ax[0].pie(counts.values, labels=counts.index, autopct="%1.0f%%",
              colors=["#4caf50", "#e05252", "#bbbbbb"][:len(counts)])
    ax[0].set_title("Results")
    ax[1].barh(by_opp["opp_archetype"], by_opp["win_rate"], color="#4a8fc8")
    ax[1].set_xlim(0, 1); ax[1].set_xlabel("win rate")
    ax[1].set_title("Win rate by opponent archetype")
    for _i, (_wr, _g) in enumerate(zip(by_opp["win_rate"], by_opp["games"])):
        ax[1].text(_wr + 0.01, _i, f"{_wr:.0%} (n={_g})", va="center", fontsize=9)
    plt.tight_layout(); plt.show()
    display(by_opp)
else:
    print("No decided games yet — download some replays first (section above).")

## 3. Episodes table

Every analyzed match, one row: result, opponent archetype (+ readable signature), opponent ELO/score (when the CLI exposes it), and per-match KPIs — turns, prizes taken/conceded, Pokémon KO'd, damage dealt/taken.

In [ ]:
cols = ["episode_id", "result", "opp_archetype", "opp_signature",
        "turns", "prizes_taken", "prizes_conceded",
        "opp_pokemon_koed", "my_pokemon_koed", "damage_dealt", "damage_taken"]
# Append any ELO/score columns merged in from the episodes listing.
cols += [c for c in kpi_df.columns if any(k in c.lower() for k in ("elo", "score", "rating"))
         and c not in cols]
table = kpi_df[[c for c in cols if c in kpi_df.columns]].copy()

def _row_color(r):
    bg = {"win": "#e7f6e7", "loss": "#fbe6e6"}.get(r["result"], "#f2f2f2")
    return [f"background-color: {bg}"] * len(r)

display(table.style.apply(_row_color, axis=1) if not table.empty else table)

## 4. Watch a replay

Set `EPISODE_ID` to any episode number (grab one from the table above), run the cell, and the full battle viewer loads inline — scrub through turns, click cards for detail, read the log. The replay JSON is downloaded on demand and cached in `./replays`.

> Tip: download the generated `replay_<id>.html` for full-screen viewing outside the notebook.

In [ ]:
from pathlib import Path

EPISODE_ID = 	84689786   # e.g. 98765432

if EPISODE_ID is None:
    print("Set EPISODE_ID above to watch a replay.")
else:
    _path = kc.get_replay_path(EPISODE_ID, dest=REPLAY_DIR, echo=False)
    print("Replay JSON:", _path)
    # Show the KPIs for this specific match, then the interactive viewer.
    display(pd.DataFrame([ra.analyze_replay_file(_path, my_team=MY_TEAM)]).T.rename(columns={0: "value"}))
    Path(HTML_DIR).mkdir(parents=True, exist_ok=True)
    _out = f"{HTML_DIR}/replay_{EPISODE_ID}.html"
    display(rr.show_replay(_path, height=900, out_path=_out))